In [ ]:

%%writefile generate_data.py

import numpy as np
import pandas as pd

# ==========================================
# GENERATOR 1: DUKA ELECTRONICS SALES DATA
# Seed: 2026
# ==========================================

rng = np.random.default_rng(2026)
n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)


# ==========================================
# GENERATOR 2: MOBILE MONEY AGENT NETWORK
# Seed: 44
# ==========================================

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20
    ),
    "onboarded": pd.date_range(
        "2024-01-01", periods=20, freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ",
         "Withdrawal", "deposit"], n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = (
    txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
)

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False),
    "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("All three datasets generated successfully.")


Writing generate_data.py


In [ ]:

!python generate_data.py


All three datasets generated successfully.


In [ ]:

import pandas as pd

sales = pd.read_csv("duka_sales.csv")
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

print("DUKA SALES")
print("Shape:", sales.shape)
display(sales.head())

print("\nAGENT REGISTER")
print("Shape:", agents.shape)
display(agents.head())

print("\nAGENT TRANSACTIONS")
print("Shape:", txns.shape)
display(txns.head())


DUKA SALES
Shape: (1000, 7)


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500



AGENT REGISTER
Shape: (20, 3)


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01



AGENT TRANSACTIONS
Shape: (612, 5)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


# Engagement 1: Duka Electronics Sales Exploration


## Executive Summary

- Nairobi was the highest-performing branch, generating **KES 26,106,100** from **376 orders**.
- Laptops generated the highest product revenue of **KES 41,535,000**, while tablets recorded the highest sales volume at **691 units**.
- Weekdays generated **KES 51,856,200** from **723 orders**, while weekend orders had a slightly higher average revenue of **KES 71,972.20** compared with **KES 71,723.65** on weekdays.
- April was the best-performing month with **KES 13,518,300**, while June was the weakest with **KES 10,903,400**, making April's revenue **23.98%** higher.
- Premium online orders from Nairobi and Mombasa totalled **83 orders**, generating **KES 10,218,000** in revenue.

## Task 1: First-Five-Minutes Dataset Inspection

The purpose of this inspection is to understand the structure, completeness, data types, date coverage and business categories before performing the sales analysis.

In [ ]:

import pandas as pd
import numpy as np

df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)


Dataset loaded successfully.
Dataset shape: (1000, 7)


In [ ]:

print("FIRST FIVE RECORDS")
display(df.head())

print("LAST FIVE RECORDS")
display(df.tail())


FIRST FIVE RECORDS


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


LAST FIVE RECORDS


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


In [ ]:

print("DATASET INFORMATION")
df.info()

print("\nMISSING VALUES")
display(df.isna().sum())

print("\nDESCRIPTIVE STATISTICS")
display(df.describe())


DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB

MISSING VALUES


,0
order_id,0
order_date,0
branch,0
product,0
quantity,0
channel,0
unit_price,0



DESCRIPTIVE STATISTICS


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


In [ ]:

print("UNIQUE VALUES PER COLUMN")
display(df.nunique())

print("\nBRANCH DISTRIBUTION")
display(df["branch"].value_counts(normalize=True).mul(100).round(2))

print("\nSALES CHANNEL DISTRIBUTION")
display(df["channel"].value_counts(normalize=True).mul(100).round(2))

print("\nPRODUCT DISTRIBUTION")
display(df["product"].value_counts())

print("\nDATE RANGE")
print("Earliest order:", df["order_date"].min())
print("Latest order:", df["order_date"].max())



UNIQUE VALUES PER COLUMN


,0
order_id,1000
order_date,179
branch,5
product,5
quantity,5
channel,3
unit_price,5



BRANCH DISTRIBUTION


,proportion
branch,
Nairobi,37.6
Mombasa,18.6
Nakuru,16.2
Kisumu,15.9
Eldoret,11.7



SALES CHANNEL DISTRIBUTION


,proportion
channel,
In-store,57.9
Online,32.0
Phone order,10.1



PRODUCT DISTRIBUTION


,count
product,
Laptop,220
Tablet,215
Phone,194
Headphones,187
Charger,184



DATE RANGE
Earliest order: 2026-01-01 00:00:00
Latest order: 2026-06-29 00:00:00



### Task 1: First-Five-Minutes Dataset Inspection — Findings

The Duka Electronics dataset contains **1,000 rows and 7 columns**, representing sales orders recorded between **1 January 2026 and 29 June 2026**.

The `order_date` column was successfully converted to datetime format, while the dataset contains three integer columns and three object columns. All seven columns have zero missing values, meaning the dataset is complete in terms of missing data.

The dataset contains five branches, five products and three sales channels. Nairobi accounts for the largest proportion of orders at **37.6%**, while Eldoret has the smallest share at **11.7%**.

In-store transactions represent the largest sales channel at **57.9%**, followed by Online at **32.0%** and Phone order at **10.1%**.

Laptops are the most frequently ordered product, with **220 orders**, followed by Tablets with 215 orders. The average quantity per order is approximately **3 units**.

**Conclusion:** The dataset is complete and has suitable data types for the planned sales exploration. No missing-value treatment is required for this dataset.



## Task 2: Creating Derived Columns

Derived columns provide additional information that helps us analyse sales performance. We will calculate revenue, extract month and weekday information, identify weekend orders and classify products into price bands.


In [ ]:

# Create the five required derived columns

# 1. Calculate total revenue for each order
df["revenue"] = df["quantity"] * df["unit_price"]

# 2. Extract the month name
df["month"] = df["order_date"].dt.month_name()

# 3. Extract the weekday name
df["weekday"] = df["order_date"].dt.day_name()

# 4. Identify whether an order was made on a weekend
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5

# 5. Classify products by unit price
df["price_band"] = np.where(
    df["unit_price"] >= 28000,
    "Premium",
    "Value"
)

print("Derived columns created successfully.")


Derived columns created successfully.


In [ ]:

# Display the original and derived columns
display(df[
    [
        "order_id",
        "order_date",
        "product",
        "quantity",
        "unit_price",
        "revenue",
        "month",
        "weekday",
        "is_weekend",
        "price_band"
    ]
].head(10))

print("\nPrice band distribution:")
display(df["price_band"].value_counts())

print("\nWeekend and weekday order counts:")
display(df["is_weekend"].value_counts())


,order_id,order_date,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Phone,3,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Laptop,4,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Tablet,5,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Headphones,5,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Headphones,3,2500,7500,March,Saturday,True,Value
5,1006,2026-03-26,Headphones,1,2500,2500,March,Thursday,False,Value
6,1007,2026-01-15,Laptop,2,65000,130000,January,Thursday,False,Premium
7,1008,2026-03-08,Charger,5,1200,6000,March,Sunday,True,Value
8,1009,2026-04-26,Headphones,5,2500,12500,April,Sunday,True,Value
9,1010,2026-03-05,Charger,5,1200,6000,March,Thursday,False,Value



Price band distribution:


,count
price_band,
Value,565
Premium,435



Weekend and weekday order counts:


,count
is_weekend,
False,723
True,277



## Task 3: Branch Performance Analysis

This task compares the five branches using total revenue and order count. The analysis will help identify the branches generating the highest sales value and those receiving the most orders.


In [ ]:

# Calculate total revenue and order count for each branch

branch_performance = df.groupby("branch").agg(
    total_revenue=("revenue", "sum"),
    order_count=("order_id", "count")
).sort_values("total_revenue", ascending=False)

print("BRANCH PERFORMANCE RANKING")
display(branch_performance)

print("\nBranch ranked by order count:")
display(branch_performance.sort_values("order_count", ascending=False))


BRANCH PERFORMANCE RANKING


,total_revenue,order_count
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117



Branch ranked by order count:


,total_revenue,order_count
branch,,
Nairobi,26106100,376
Mombasa,14003700,186
Nakuru,11970400,162
Kisumu,10868100,159
Eldoret,8844200,117



### Task 3: Branch Performance — Findings

Nairobi is the best-performing branch, generating total revenue of **KES 26,106,100** from **376 orders**. It ranks first in both total revenue and order count.

Mombasa ranks second, generating **KES 14,003,700** from 186 orders. Nakuru follows with KES 11,970,400 from 162 orders.

Kisumu generated KES 10,868,100 from 159 orders, while Eldoret recorded the lowest performance, with KES 8,844,200 from 117 orders.

**Business insight:** Nairobi is the strongest branch in terms of both sales value and order volume. Eldoret has the lowest revenue and order count, suggesting that management could investigate opportunities to improve its sales performance.



## Task 4: Product Performance Analysis

This task identifies the top three products by total revenue and total units sold. Comparing the two rankings helps explain the difference between sales value and product demand.


In [ ]:

# Calculate product revenue and units sold

product_performance = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    units_sold=("quantity", "sum")
)

print("TOP THREE PRODUCTS BY REVENUE")
display(
    product_performance
    .sort_values("total_revenue", ascending=False)
    .head(3)
)

print("\nTOP THREE PRODUCTS BY UNITS SOLD")
display(
    product_performance
    .sort_values("units_sold", ascending=False)
    .head(3)
)


TOP THREE PRODUCTS BY REVENUE


,total_revenue,units_sold
product,,
Laptop,41535000,639
Tablet,19348000,691
Phone,8880000,592



TOP THREE PRODUCTS BY UNITS SOLD


,total_revenue,units_sold
product,,
Tablet,19348000,691
Laptop,41535000,639
Phone,8880000,592



### Task 4: Product Performance — Findings

Laptops generated the highest revenue of **KES 41,535,000**, despite selling 639 units. Tablets ranked second in revenue, generating **KES 19,348,000**, but recorded the highest sales volume at **691 units**.

Phones ranked third in both revenue, at **KES 8,880,000**, and units sold, at **592 units**.

The difference between revenue and units sold is caused by differences in product prices. Laptops generate more revenue per unit because their unit price is higher, while tablets have greater sales volume.

**Business insight:** Duka should continue monitoring laptop revenue because of its high contribution to sales value, while maintaining sufficient tablet stock because tablets have the highest unit demand.



## Task 5: Weekend versus Weekday Sales Analysis

This task compares weekday and weekend sales performance using order count, total revenue and average revenue per order. The results will help determine whether weekend or weekday transactions contribute more to business performance.


In [ ]:

# Compare weekday and weekend sales

weekend_analysis = df.groupby("is_weekend").agg(
    order_count=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

# Give the results readable labels
weekend_analysis.index = ["Weekday", "Weekend"]

print("WEEKDAY VS WEEKEND SALES PERFORMANCE")
display(weekend_analysis.round(2))


WEEKDAY VS WEEKEND SALES PERFORMANCE


,order_count,total_revenue,average_order_revenue
Weekday,723,51856200,71723.65
Weekend,277,19936300,71972.20



### Task 5: Weekend versus Weekday — Findings

Weekday sales recorded **723 orders**, generating total revenue of **KES 51,856,200**. Weekend sales recorded **277 orders**, generating **KES 19,936,300**.

Weekdays generated more total revenue because they had significantly more orders. However, the average revenue per order was slightly higher on weekends at **KES 71,972.20**, compared with **KES 71,723.65** on weekdays.

**Business insight:** Weekdays contribute more to overall revenue because of their higher transaction volume. However, weekend customers spend slightly more per order on average. Duka could maintain strong weekday operations while exploring promotions to attract more weekend customers.



## Task 6: Monthly Revenue Analysis

This task calculates total revenue for each month, identifies the best and worst performing months, and calculates the percentage difference between their revenues.


In [ ]:

# Calculate total revenue for each month

monthly_revenue = df.groupby("month")["revenue"].sum()

# Arrange months in calendar order
month_order = [
    "January", "February", "March",
    "April", "May", "June"
]

monthly_revenue = monthly_revenue.reindex(month_order)

print("MONTHLY REVENUE")
display(monthly_revenue.to_frame(name="total_revenue"))

# Identify the best and worst months
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

# Calculate percentage difference relative to the worst month
percentage_difference = (
    (best_revenue - worst_revenue) / worst_revenue
) * 100

print("\nBest month:", best_month)
print("Best month revenue: KES", best_revenue)

print("\nWorst month:", worst_month)
print("Worst month revenue: KES", worst_revenue)

print("\nPercentage difference: {:.2f}%".format(
    percentage_difference
))


MONTHLY REVENUE


,total_revenue
month,
January,11786600
February,11772000
March,12396800
April,13518300
May,11415400
June,10903400



Best month: April
Best month revenue: KES 13518300

Worst month: June
Worst month revenue: KES 10903400

Percentage difference: 23.98%



### Task 6: Monthly Revenue — Findings

April was the best-performing month, generating total revenue of **KES 13,518,300**. June was the worst-performing month, generating **KES 10,903,400**.

The difference between the best and worst months was KES 2,614,900. April's revenue was **23.98% higher** than June's revenue.

March also performed well, generating KES 12,396,800, while May recorded KES 11,415,400.

**Business insight:** April had the strongest sales performance, while June had the weakest. Management should investigate the factors behind the June decline and consider repeating successful sales strategies used in April.



## Task 7: Premium Online Sales in Nairobi and Mombasa

This task filters the sales dataset to identify Premium orders placed through the Online channel in Nairobi and Mombasa. We will calculate the number of matching orders, their total revenue and export the filtered records to a CSV file.


In [ ]:

# Filter Premium Online orders from Nairobi and Mombasa

premium_online_coast_capital = df[
    (df["price_band"] == "Premium") &
    (df["channel"] == "Online") &
    (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

# Calculate order count and total revenue
filtered_order_count = len(premium_online_coast_capital)
filtered_total_revenue = premium_online_coast_capital["revenue"].sum()

print("PREMIUM ONLINE ORDERS: NAIROBI AND MOMBASA")
print("Number of orders:", filtered_order_count)
print("Total revenue: KES", filtered_total_revenue)

# Display the filtered records
display(premium_online_coast_capital.head(10))

# Export without the DataFrame index
premium_online_coast_capital.to_csv(
    "premium_online_coast_capital.csv",
    index=False
)

print("\nCSV file exported successfully.")


PREMIUM ONLINE ORDERS: NAIROBI AND MOMBASA
Number of orders: 83
Total revenue: KES 10218000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
6,1007,2026-01-15,Nairobi,Laptop,2,Online,65000,130000,January,Thursday,False,Premium
13,1014,2026-06-12,Nairobi,Tablet,3,Online,28000,84000,June,Friday,False,Premium
32,1033,2026-04-29,Nairobi,Laptop,2,Online,65000,130000,April,Wednesday,False,Premium
35,1036,2026-03-02,Nairobi,Laptop,4,Online,65000,260000,March,Monday,False,Premium
39,1040,2026-02-10,Mombasa,Laptop,5,Online,65000,325000,February,Tuesday,False,Premium
48,1049,2026-03-25,Mombasa,Tablet,1,Online,28000,28000,March,Wednesday,False,Premium
58,1059,2026-04-30,Mombasa,Laptop,1,Online,65000,65000,April,Thursday,False,Premium
77,1078,2026-06-12,Nairobi,Tablet,2,Online,28000,56000,June,Friday,False,Premium
82,1083,2026-03-16,Nairobi,Tablet,3,Online,28000,84000,March,Monday,False,Premium
103,1104,2026-03-08,Mombasa,Laptop,3,Online,65000,195000,March,Sunday,True,Premium



CSV file exported successfully.



### Task 7: Premium Online Sales — Findings

The analysis identified **83 Premium online orders** from the Nairobi and Mombasa branches. These orders generated total revenue of **KES 10,218,000**.

The filter correctly selected orders where the price band was Premium, the sales channel was Online and the branch was either Nairobi or Mombasa.

**Business insight:** Premium online sales provide a useful opportunity for targeted marketing in the two major branches. Duka can use these findings to plan online promotions and monitor demand for higher-priced products.


In [ ]:

import os

# Create outputs folder if it does not exist
os.makedirs("outputs", exist_ok=True)

# Save the filtered dataset in the required location
premium_online_coast_capital.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

# Verify the exported file
print("File exists:",
      os.path.exists("outputs/premium_online_coast_capital.csv"))

print("Exported rows:",
      pd.read_csv("outputs/premium_online_coast_capital.csv").shape[0])


File exists: True
Exported rows: 83


In [ ]:

# Prepare the spreadsheet source data
sheets_data = df[
    [
        "order_id",
        "order_date",
        "branch",
        "product",
        "quantity",
        "channel",
        "unit_price",
        "revenue"
    ]
].copy()

# Export the data without the DataFrame index
sheets_data.to_csv("duka_sales_for_sheets.csv", index=False)

print("Spreadsheet data prepared:", sheets_data.shape)


Spreadsheet data prepared: (1000, 8)


In [ ]:
from google.colab import files
files.download("duka_sales_for_sheets.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from google.colab import files

uploaded = files.upload()


Saving Duka Branch Sales Analysis.xlsx to Duka Branch Sales Analysis.xlsx


In [19]:
import os
import shutil

# Create the excel folder if it does not exist
os.makedirs("excel", exist_ok=True)

# Get the uploaded filename automatically
uploaded_filename = next(iter(uploaded))

# Move and rename the workbook
shutil.move(
    uploaded_filename,
    "excel/duka_branch_summary.xlsx"
)

print("Excel workbook saved successfully!")

Excel workbook saved successfully!


In [20]:
import os

file_path = "excel/duka_branch_summary.xlsx"

print("File exists:", os.path.exists(file_path))
print("File size:", os.path.getsize(file_path), "bytes")


File exists: True
File size: 73615 bytes


In [21]:
from google.colab import files

files.download("outputs/premium_online_coast_capital.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
files.download("excel/duka_branch_summary.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>